# Lab 03: Backpropagation and Gradient Diagnostics

            **Duration:** 3 hours  
            **Lecture alignment:** Week 3 — Backpropagation and training diagnostics  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Verify chain-rule gradients against finite differences.
- Instrument layerwise gradient norms to identify vanishing or exploding gradients.
- Apply clipping and defensible debugging checks.

            ## Three-hour activity plan

            - 0–35 min: chain-rule derivation and numerical check
- 35–75 min: hooks/gradient inspection
- 75–125 min: sigmoid versus ReLU deep-network diagnostic
- 125–160 min: clipped training and anomaly discussion
- 160–180 min: evidence capture and reflection


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20263
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_03")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_03"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 3, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Predict which network, deep sigmoid or deep ReLU, will have the smaller gradient in its earliest layer. What would count as convincing evidence?

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Chain rule and finite-difference diagnostics


In [ ]:
x = torch.tensor(1.25, requires_grad=True)
y = torch.sin(x**2) + 0.5*x
y.backward(); auto_grad = x.grad.item()
eps = 1e-4
fn = lambda z: torch.sin(z**2) + .5*z
numeric_grad = ((fn(x.detach()+eps) - fn(x.detach()-eps))/(2*eps)).item()
relative_error = abs(auto_grad-numeric_grad)/(abs(numeric_grad)+1e-8)
print({"autograd": auto_grad, "numeric": numeric_grad, "relative_error": relative_error})


## Activity 2 — Diagnose gradients in shallow and deep networks


In [ ]:
n = 700 if FAST_MODE else 3000
X = torch.randn(n, 8)
teacher = torch.tensor([1.0, -1.0, .6, -.6, .4, -.4, .2, -.2])
y = ((X @ teacher + .3*torch.randn(n)) > 0).long()
split = int(.8*n); Xtr, Xte, ytr, yte = X[:split], X[split:], y[:split], y[split:]

class DeepNet(nn.Module):
    def __init__(self, activation="relu"):
        super().__init__()
        act = nn.ReLU if activation == "relu" else nn.Sigmoid
        layers = []
        for i in range(6):
            layers += [nn.Linear(8 if i == 0 else 24, 24), act()]
        self.features = nn.Sequential(*layers); self.head = nn.Linear(24, 2)
    def forward(self, x): return self.head(self.features(x))

def one_step_gradient_report(activation):
    torch.manual_seed(SEED)
    model = DeepNet(activation).to(DEVICE)
    loss = F.cross_entropy(model(Xtr.to(DEVICE)), ytr.to(DEVICE)); loss.backward()
    report = {name: p.grad.norm().item() for name, p in model.named_parameters() if p.grad is not None and "weight" in name}
    return model, loss.item(), report

sigmoid_model, sigmoid_loss, sigmoid_grads = one_step_gradient_report("sigmoid")
relu_model, relu_loss, relu_grads = one_step_gradient_report("relu")
labels = list(sigmoid_grads)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.semilogy([sigmoid_grads[k]+1e-12 for k in labels], "o-", label="sigmoid")
ax.semilogy([relu_grads[k]+1e-12 for k in labels], "o-", label="ReLU")
ax.set(xticks=range(len(labels)), xticklabels=labels, ylabel="gradient norm", title="Layerwise gradient diagnostics")
ax.tick_params(axis="x", rotation=45); ax.legend(); fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "gradient_norms.png", dpi=150); plt.show()


## Activity 3 — Gradient clipping and anomaly-safe training


In [ ]:
model = DeepNet("relu").to(DEVICE)
opt = torch.optim.SGD(model.parameters(), lr=.08)
history, clipped_norms = [], []
for _ in range(20 if FAST_MODE else 70):
    opt.zero_grad(); loss = F.cross_entropy(model(Xtr.to(DEVICE)), ytr.to(DEVICE)); loss.backward()
    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    opt.step(); history.append(loss.item()); clipped_norms.append(float(total_norm))
with torch.no_grad(): acc = (model(Xte.to(DEVICE)).argmax(1).cpu() == yte).float().mean().item()
print({"test_accuracy": acc, "initial_loss": history[0], "final_loss": history[-1], "max_preclip_norm": max(clipped_norms)})


## Automated checks


In [ ]:
assert relative_error < 2e-3
assert all(math.isfinite(v) for v in sigmoid_grads.values())
assert history[-1] < history[0] and acc > .70
assert (ARTIFACT_DIR / "gradient_norms.png").exists()
print("All Lab 03 checks passed.")


## Deliverables

                - Finite-difference verification
- Layerwise gradient-norm visualization
- Clipped training run and diagnosis of one failure mode

                Submit the executed notebook and the files created in `/content/artifacts/lab_03/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    with torch.autograd.detect_anomaly():
        z = torch.tensor(-1.0, requires_grad=True); bad = torch.sqrt(z); bad.backward()
else:
    print("Extension disabled: use anomaly detection to localize an invalid backward operation.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
